In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix,classification_report,f1_score,roc_auc_score, accuracy_score

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
df=pd.read_csv('/Users/siva/Downloads/digital_marketing_campaign_dataset.csv')

In [4]:
df.head()

,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool,Conversion
0,8000,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,9,4,688,IsConfid,ToolConfid,1
1,8001,69,Male,41760,Email,Retention,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,7,2,3459,IsConfid,ToolConfid,1
2,8002,46,Female,88456,PPC,Awareness,1546.429596,0.277490,0.076423,2,8.223619,13.794901,0,11,2,8,2337,IsConfid,ToolConfid,1
3,8003,32,Female,44085,PPC,Conversion,539.525936,0.137611,0.088004,47,4.540939,14.688363,89,2,2,0,2463,IsConfid,ToolConfid,1
4,8004,60,Female,83964,PPC,Conversion,1678.043573,0.252851,0.109940,0,2.046847,13.993370,6,6,6,8,4345,IsConfid,ToolConfid,1


In [5]:
# Train test split

independent = df.drop('Conversion', axis=1)
dependent = df['Conversion']

In [6]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical variables before scaling
# Create label encoders for categorical columns
label_encoders = {}

# Identify categorical columns (assuming they contain string/object data)
categorical_columns = independent.select_dtypes(include=['object']).columns

# Encode each categorical column
indep_encoded = independent.copy()
for column in categorical_columns:
    le = LabelEncoder()
    indep_encoded[column] = le.fit_transform(independent[column])
    label_encoders[column] = le  # Store encoder for future use

# train_test_split data with encoded features
X_train, X_test, y_train, y_test = train_test_split(indep_encoded, dependent, test_size=0.2, random_state=0)

# Standard scaler - now works with numeric data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
param_grid={'criterion':['gini', 'entropy', 'log_loss'],'max_features': ['auto','sqrt','log2'],
              'n_estimators':[10,100]}
grid=GridSearchCV(RandomForestClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1')

# fitting the model for grid search 
grid.fit(X_train, y_train)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


,estimator,RandomForestClassifier()
,param_grid,"{'criterion': ['gini', 'entropy', ...], 'max_features': ['auto', 'sqrt', ...], 'n_estimators': [10, 100]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,10


In [9]:
re=grid.cv_results_
grid_predictions = grid.predict(X_test_scaled) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
roc_score = roc_auc_score(y_test,grid.predict_proba(X_test_scaled)[:,1])
#accuracy
accuracy = accuracy_score(y_test, grid_predictions)

In [10]:
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("\nThe confusion Matrix:\n",cm)
print("\nThe report:\n",clf_report)
print("\nROC_AUC_Score:",roc_score)
print("\nAccuracy score:",accuracy)

The f1_macro value for best parameter {'criterion': 'log_loss', 'max_features': 'sqrt', 'n_estimators': 10}: 0.0627935606060606

The confusion Matrix:
 [[ 190    4]
 [1376   30]]

The report:
               precision    recall  f1-score   support

           0       0.12      0.98      0.22       194
           1       0.88      0.02      0.04      1406

    accuracy                           0.14      1600
   macro avg       0.50      0.50      0.13      1600
weighted avg       0.79      0.14      0.06      1600


ROC_AUC_Score: 0.49459056180434363

Accuracy score: 0.1375
